# Solutions · Chapter 04-05 · Leakage lab

Worked answers for `notebooks/04_workflow/04-05_leakage.ipynb`.

E9, E10 and E11 each isolate one part of the severity principle, and E9's answer runs the opposite way
to most people's intuition.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
folds = StratifiedKFold(5, shuffle=True, random_state=0)

# SYNTHETIC: the chapter's pure-noise dataset.
noise_rng = np.random.default_rng(0)
noise = noise_rng.normal(size=(200, 5000))
coin_flip = noise_rng.integers(0, 2, 200)


# SYNTHETIC: 04-01's gym panel, framed with the two leaky columns.
def load_members():
    rng = np.random.default_rng(41)
    n = 600
    join_month = rng.integers(1, 13, n)
    commitment = rng.beta(2.0, 2.0, n)
    rows = []
    for member in range(n):
        drifting = 0.0
        base_visits = 2 + 10 * commitment[member]
        for month in range(join_month[member], 25):
            drifting += rng.normal(0.0, 0.35)
            visits = max(0, int(round(rng.poisson(max(0.2, base_visits - drifting)))))
            tickets = int(rng.random() < 0.05 + 0.10 * (visits == 0))
            hazard = 1 / (1 + np.exp(3.0 + 2.5 * commitment[member] - 0.55 * max(0, 4 - visits)))
            cancelled = int(rng.random() < hazard)
            rows.append((member + 1, month, visits, tickets, cancelled))
            if cancelled:
                break
    return pd.DataFrame(rows, columns=["member_id", "month", "visits", "tickets", "cancelled"])


panel = load_members().sort_values(["member_id", "month"])
CUT, HORIZON = 12, 6
active = panel[(panel.month == CUT) & (panel.cancelled == 0)].member_id.unique()
history = panel[panel.member_id.isin(active) & (panel.month <= CUT)]
future = panel[panel.member_id.isin(active) & (panel.month > CUT)]
left = future[(future.month <= CUT + HORIZON) & (future.cancelled == 1)].member_id.unique()
churn = pd.Series(np.isin(active, left).astype(int), index=active)

whole = panel[panel.member_id.isin(active)]
leaky_churn = pd.DataFrame({
    "visits_at_cut": history[history.month == CUT].set_index("member_id").visits.reindex(active),
    "mean_visits_last_3": history[history.month > CUT - 3].groupby("member_id").visits.mean().reindex(active),
    "tenure_months": history.groupby("member_id").size().reindex(active),
    "tickets_so_far": history.groupby("member_id").tickets.sum().reindex(active),
    "visits_lifetime": whole.groupby("member_id").visits.sum().reindex(active),
    "months_on_file": whole.groupby("member_id").size().reindex(active),
})
print("noise data %s; leaky churn table %s" % (noise.shape, leaky_churn.shape))

## Quick understanding

### E1

| Kind | What moves, and from where |
|---|---|
| **Target leakage** | the answer itself, from the label into a feature column |
| **Duplicate / group leakage** | a test row's label, from the test set into training, via a copy or a repeated entity |
| **Temporal leakage** | information from after the prediction moment, from the future into the past |
| **Preprocessing leakage** | a summary of the test rows, from the test set into a fitted transformation |

### E2

**Damage is proportional to how much information about the target the fitted step absorbed.**

`PolynomialFeatures` fitted before the split is **not dangerous**. It never sees `y` - it only records how
many columns there are and generates products of them - so it has nothing about the target to leak. It is
in the same family as a scaler.

(It is still worth putting in the pipeline, for the reason the chapter gave: cheapness and habit, not
fear. And note the different problem it does create - it multiplies the column count, which makes any
*subsequent* selection step far more dangerous.)

### E3

Because cross-validation faithfully repeats **whatever procedure it is given**, and the corruption
happened before it started. The 20 columns were already chosen using all 200 rows, so every fold's
"held-out" rows had already contributed to deciding which columns the model would see. Splitting after
that point cannot undo it.

The general form: **cross-validation validates the step you put inside it.** Anything fitted outside is
simply not being validated.

## Hand calculation

### E4

- **250 categories over 500 rows:** about 2 rows each, so a row's own label is **1/2 = 50%** of its
  encoded value.
- **25 categories over 500 rows:** 20 rows each, so **1/20 = 5%**.

**The rule: risk is governed by rows *per category*, not by the number of categories.** A column with a
thousand categories and a million rows is safe; a column with fifty categories and a hundred rows is not.
The quantity to compute before target-encoding anything is `len(frame) / frame[column].nunique()`, and
anything in single figures should be treated as a copy of the label.

Ids, postcodes, product codes and free-text categories are the usual offenders, because their cardinality
grows with the dataset - so the ratio does **not** improve as you collect more data.

### E5

Each correlation has a standard deviation of about `1/sqrt(200)` = 0.071. The expected maximum of 5,000
standard normal draws is about 3.6, so the largest correlation will be around

`3.6 x 0.071` = **0.255**

**A selector will therefore find columns correlating about 0.25 with the target, in data where the true
correlation is exactly zero.** That is a respectable-looking correlation - the kind that would survive a
scatter plot and a significance test done naively - and there are thousands of columns competing to
produce it. 02-07's multiple-comparisons warning, now with the arithmetic attached.

### E6

Symptom 1 (**better than the problem allows**) is triggered: 0.96 against a published best of 0.83.
Symptom 3 (**folds agree too well**) is triggered: a standard deviation of 0.004 is implausibly tight.

Symptom 2 (**one feature dominates**) is *not* assessed - the question gives no information about
features, and it is the one check you have to run rather than read off.

Two triggered symptoms is not proof, and the correct next step is symptom 2 plus the two questions. But
"far above the field, with unusually stable folds" is the exact fingerprint of a systematic advantage
present in every fold.

### E7

180 rows are involved in a duplicate pair (90 originals plus 90 copies). Each is assigned independently,
so for any one of them: `P(this row in test) x P(its twin in training)` = `0.2 x 0.8` = 0.16.

Expected number: `180 x 0.16` = **28.8 test rows whose twin is sitting in the training set.**

With 78 test rows in total (20% of 390), that is over a third of the test set answerable by lookup.

In [ ]:
print("E4  own share of the encoding: 250 categories -> %.0f%%, 25 categories -> %.0f%%"
      % (100 / 2, 100 / 20))
print("E5  expected largest correlation among 5,000: 3.6 / sqrt(200) = %.3f" % (3.6 / np.sqrt(200)))
print("E7  expected leaked test rows: 180 x 0.2 x 0.8 = %.1f  (of %d test rows)"
      % (180 * 0.2 * 0.8, round(0.2 * 390)))

## Coding

### E8 - a leakage report

In [ ]:
def leakage_report(frame, entity_column, time_column):
    print("rows %d, columns %d" % frame.shape)

    duplicates = frame.duplicated().sum()
    print("  exact duplicate rows        : %d %s"
          % (duplicates, "" if duplicates == 0 else "<- investigate before splitting"))

    counts = frame[entity_column].value_counts()
    repeated = int((counts > 1).sum())
    print("  distinct %-19s: %d" % (entity_column, len(counts)))
    print("  appearing more than once    : %d (%.1f%%), up to %d times"
          % (repeated, 100 * repeated / len(counts), counts.max()))
    if repeated:
        print("      -> a random split will put the same entity on both sides. Use GroupKFold.")

    print("  %s spans %s to %s"
          % (time_column, frame[time_column].min(), frame[time_column].max()))
    if frame[time_column].nunique() > 1:
        print("      -> if predictions are made forward in time, split chronologically.")


def frame_at(cut, horizon=6):
    active_now = panel[(panel.month == cut) & (panel.cancelled == 0)].member_id.unique()
    if len(active_now) == 0:
        return None
    past = panel[panel.member_id.isin(active_now) & (panel.month <= cut)]
    ahead = panel[panel.member_id.isin(active_now) & (panel.month > cut)]
    gone = ahead[(ahead.month <= cut + horizon) & (ahead.cancelled == 1)].member_id.unique()
    built = pd.DataFrame({
        "visits_now": past[past.month == cut].set_index("member_id").visits.reindex(active_now),
        "tenure_months": past.groupby("member_id").size().reindex(active_now)})
    built["member_id"] = active_now
    built["cut_month"] = cut
    built["target"] = np.isin(active_now, gone).astype(int)
    return built.reset_index(drop=True)


stacked = pd.concat([frame_at(cut) for cut in range(1, 19)], ignore_index=True)
leakage_report(stacked, "member_id", "cut_month")

### E9 - does keeping more columns help or hurt?

In [ ]:
rows = []
for keep in [1, 5, 20, 100]:
    selected = SelectKBest(f_classif, k=keep).fit(noise, coin_flip)
    leaky = cross_val_score(LogisticRegression(max_iter=2000), selected.transform(noise),
                            coin_flip, cv=folds, scoring="accuracy").mean()
    honest = cross_val_score(make_pipeline(SelectKBest(f_classif, k=keep),
                                           LogisticRegression(max_iter=2000)),
                             noise, coin_flip, cv=folds, scoring="accuracy").mean()
    rows.append({"columns kept": keep, "leaky accuracy": round(leaky, 4),
                 "honest accuracy": round(honest, 4), "inflation": round(leaky - honest, 4)})
keep_table = pd.DataFrame(rows)
print(keep_table.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 4.3))
ax.plot(keep_table["columns kept"], keep_table["leaky accuracy"], "o-", color="#D55E00",
        label="selected before splitting")
ax.plot(keep_table["columns kept"], keep_table["honest accuracy"], "s-", color="#0072B2",
        label="selected inside the pipeline")
ax.axhline(0.5, color="#000000", linestyle=":", linewidth=1.4, label="the truth")
ax.set_xscale("log")
ax.set_xlabel("columns kept by the selector (log scale)")
ax.set_ylabel("reported accuracy")
ax.set_title("Keeping more pre-selected noise columns makes it worse, not better", fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**Worse - much worse. Keeping 100 columns reports 96.5% accuracy on a coin flip.**

That is the opposite of most people's intuition, which says that keeping more columns dilutes the few
lucky ones. The reason it fails:

**Every one of the 100 kept columns was chosen for correlating with the target across all 200 rows** -
including the held-out ones. So each carries a small, *genuine-within-this-dataset* association with the
label, and the association is present in the test fold too, because the test fold is where it was
measured from. Give a logistic regression 100 such columns and it combines 100 weak leaks into a strong
one.

With `k = 1` the model has a single lucky column and reports 66%. With `k = 100` it has a hundred, and
they stack.

**The honest line is flat around 0.5 to 0.6 regardless of `k`**, as it must be - inside the pipeline the
selector is choosing from the training fold only, and its choices do not transfer.

The transferable warning: **"I kept a lot of features, so no single one can be driving it" is not a
defence against selection leakage.** It is an aggravating factor.

### E10 - does smoothing fix a leaky target encoding?

In [ ]:
encode_rng = np.random.default_rng(3)
category = encode_rng.integers(0, 150, 600)
label = encode_rng.integers(0, 2, 600)
overall = label.mean()

rows = []
for smoothing in [0, 1, 5, 20, 100]:
    stats = pd.Series(label).groupby(category).agg(["mean", "size"])
    smoothed = (stats["mean"] * stats["size"] + overall * smoothing) / (stats["size"] + smoothing)
    encoded = pd.Series(category).map(smoothed).to_numpy().reshape(-1, 1)
    rows.append({"smoothing weight": smoothing,
                 "AUC (still computed on all rows)":
                     round(cross_val_score(LogisticRegression(max_iter=2000), encoded, label,
                                           cv=folds, scoring="roc_auc").mean(), 4)})
print(pd.DataFrame(rows).to_string(index=False))
print()
print("the truth: 0.5000")

**No. Smoothing barely touches it** - 0.7659 unsmoothed, 0.7503 with a weight of 100, which is heavier
smoothing than anybody would use on 4-row categories.

**What this proves is where the problem lives.** Smoothing is a fix for *variance*: it stops a category
with two rows from producing a wildly overconfident estimate. It has nothing to say about *whose labels*
went into the estimate, and that is the leak. A smoothed average of a row's own label is still an average
of a row's own label.

The fix is structural, not statistical: **compute the encoding inside the fold, from training rows only.**
Then smoothing becomes what it was meant to be - a useful regulariser on a legitimate feature. Both are
worth having; only one of them is the fix.

### E11 - which step has to be inside the pipeline?

In [ ]:
with_gaps = noise.copy()
with_gaps[noise_rng.random(with_gaps.shape) < 0.05] = np.nan

everything_inside = make_pipeline(SimpleImputer(), StandardScaler(),
                                  SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))

prepared = StandardScaler().fit_transform(SimpleImputer().fit_transform(with_gaps))
selector_inside = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))
selected_outside = SelectKBest(f_classif, k=20).fit(prepared, coin_flip)

print("everything inside the pipeline          : %.4f"
      % cross_val_score(everything_inside, with_gaps, coin_flip, cv=folds, scoring="accuracy").mean())
print("impute and scale outside, select inside : %.4f"
      % cross_val_score(selector_inside, prepared, coin_flip, cv=folds, scoring="accuracy").mean())
print("selector outside as well                : %.4f"
      % cross_val_score(LogisticRegression(max_iter=2000), selected_outside.transform(prepared),
                        coin_flip, cv=folds, scoring="accuracy").mean())
print("the truth                               : 0.5000")

**Only the selector matters.** Moving the imputer and the scaler outside changes the answer from 0.5550
to 0.5600 - nothing. Moving the *selector* outside takes it to 0.8150.

This is the severity principle demonstrated by dissection rather than asserted: the two steps that never
look at `y` can be moved anywhere without consequence, and the one that does cannot.

**The practical instruction remains "put them all inside"**, because the cost is a single line and it
removes the need to make this judgement correctly every time, on every project, under deadline. But
knowing *which* step was doing the damage is what lets you triage a legacy codebase in an hour instead of
a week.

### E12 - drop each feature in turn

In [ ]:
def drop_one_feature_report(features, target, scoring="roc_auc"):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    baseline = cross_val_score(model, features, target, cv=folds, scoring=scoring).mean()
    rows = []
    for column in features.columns:
        without = cross_val_score(model, features.drop(columns=[column]), target,
                                  cv=folds, scoring=scoring).mean()
        alone = cross_val_score(model, features[[column]], target, cv=folds, scoring=scoring).mean()
        rows.append({"feature": column, "score without it": round(without, 4),
                     "drop": round(baseline - without, 4), "score from it alone": round(alone, 4)})
    print("all features together: %.4f" % baseline)
    return pd.DataFrame(rows).sort_values("drop", ascending=False)


print(drop_one_feature_report(leaky_churn, churn).to_string(index=False))

**The exercise asked for a report that identifies `months_on_file`, and the drop column does not.** It
ranks `tenure_months` first, at 0.0492, with `months_on_file` second at 0.0252 - so the largest drop
belongs to the innocent column. That is worth understanding rather than patching over, because it
generalises.

**Drop-one-feature is confounded by redundancy.** Removing a column only hurts if nothing else can
replace it. `months_on_file` can be largely reconstructed from `visits_lifetime`, which is also
whole-history, so dropping it costs little. `tenure_months` is the only column carrying the join date,
and 04-01 showed that `months_on_file + 12 - tenure_months` is the exact cancellation month - so removing
`tenure_months` destroys the decoder, and the leak with it.

**Read the last column instead.** "Score from it alone" fingers the culprits correctly: `months_on_file`
scores **0.9323** by itself and `visits_lifetime` **0.9153**, against 0.5279 for `tenure_months`. A single
column that nearly reproduces the whole model is either the entire signal or a copy of the answer, and on
a hard problem the second is far more likely.

So the two diagnostics answer different questions, and the leak-hunting one is the second:

- **"Drop"** answers *how much does the model rely on this column, given the others* - useful for
  simplifying a model, misleading for finding leaks, because redundant leaks hide each other.
- **"Alone"** answers *how much does this column know about the target* - which is the leakage question.

And it still does not catch everything: the *pair* is what reconstructs the answer, and no per-column
statistic can see a pair. **Per-column diagnostics catch per-column leaks**, which is the same limitation
04-01's detector had.

## Interpretation

### E13

**The case that it is legitimate:** unhappy customers phone. Complaints genuinely precede cancellation,
they are recorded at the time they happen, and a churn model that uses them is using a real leading
indicator. Retention teams have acted on exactly this signal for decades.

**The case that it is leakage:** the *last* call is very often the cancellation call itself. If the count
includes calls up to the moment of cancellation, then a customer who cancelled has, by construction, at
least one more call than they otherwise would - and the feature partly counts the event it is predicting.

**The single fact that settles it: what is the timestamp of the last call included in the count, relative
to the prediction moment?** If the feature is "calls in the 90 days before the prediction date" it is
legitimate. If it is "calls, total, as recorded in the current database" it is `months_on_file` again.

That is 04-01's availability question, and it is answered by looking at the feature-building code, not by
arguing about plausibility.

### E14

**The likely mechanism:** departments see different patients. An oncology department has a far higher rate
of cancer diagnoses than a general clinic, so `hospital_department` is a proxy for the base rate - and
knowing the base rate of the group a patient was routed into is enormously predictive without being
diagnostic at all. The model has learned the hospital's triage process.

**Why it may still be useful:** triage information is real and available at prediction time. If the model
runs after a patient is routed, the department is a legitimate feature and the model is genuinely helpful
within that hospital. It is not leakage in the strict sense - the value *is* known when the prediction is
made.

**Why it would fail elsewhere:** another hospital's department names, referral patterns and case mixes are
different. The model has learned a mapping from department to base rate that is a property of *this*
institution. Transported, it applies the wrong priors confidently - and the failure is silent, because
the column exists and contains plausible values.

This is the difference between **leakage** (unavailable at prediction time) and **a shortcut** (available,
predictive, and not the thing you meant to learn). The second is not a bug, but it constrains where the
model may be deployed, and that constraint belongs in the documentation.

## Debugging

### E15

1. **A feature computed from the target elsewhere in the pipeline** - a target encoding, a feature
   selected on all data, an aggregate that includes the label, or a column somebody helpfully
   back-filled. The split cannot protect you from a column that already contains the answer.
2. **The label itself is contaminated.** If the label was derived from a field that also feeds a feature -
   a status column updated when the event happens, or a label defined by a rule that uses one of your
   inputs - the model is partly reproducing a definition.
3. **The entity id is not what you think it is.** Grouping by `customer_id` does nothing if the same
   person has three ids, if households share behaviour, or if the id is regenerated per session. Check
   for near-duplicates across the split boundary, not just id equality.

A fourth, worth mentioning: **the test set is not representative** - it may be trivially easy for a reason
unrelated to leakage. Check its base rate and its feature distributions against the training set.

### E16

**Report 0.71.** It is the score of a procedure that could actually be run at prediction time; 0.94 is the
score of a procedure that cannot.

**What to investigate next, because 0.71 is not necessarily the end:**

- **Which step caused the drop?** Move them back out one at a time (E11). If it was the scaler, something
  odd is happening and it is worth understanding. If it was a selector or an encoder, the drop is expected
  and the matter is closed.
- **Is 0.71 above the baseline?** 04-02's question, and it is now the important one - a correct 0.71 may
  or may not beat a per-entity mean or a single threshold.
- **Was the 0.94 procedure ever *used* to make decisions?** If features or hyperparameters were chosen
  while the leak was in place, those choices were made on corrupted evidence and should be revisited.
  The score is the visible damage; the choices are the lasting one.

## Exam and interview reasoning

### E17

> "Leakage is information reaching the model that will not exist when it runs for real. There are four
> kinds: a feature that is a disguised copy of the target; the same entity or a duplicate row on both
> sides of the split; training on data from after the prediction moment; and a preprocessing step fitted
> before the split. The way to see how bad it gets: take 200 rows, 5,000 columns of pure random noise and
> a coin-flip target, select the 20 best columns using all the data, then cross-validate - it reports 79%
> accuracy on something that is 50% by construction. Prevention is structural: every fitted step inside a
> Pipeline so it is refitted per fold, plus assertions that no entity and no time period crosses the
> split."

**"We cross-validate everything, so we are covered, right?"**

> "That 79% number *was* five-fold cross-validated, and the folds agreed to within a few percent. Cross-
> validation only validates what is inside it - the selection happened before, so the folds were repeating
> an already-corrupted procedure. It is also worth knowing that tight folds are mildly *evidence for*
> leakage rather than against it, because a systematic advantage shows up equally in every fold: it raises
> the mean and lowers the variance at the same time."

## Transfer to a different situation

### E18

Predicting whether an applicant will be hired, from their CV:

| Kind | The leak | How to check |
|---|---|---|
| **Target** | a field written during or after the decision - recruiter notes, an "offer date", a status code, or a CV that was reformatted after hiring | list every column with its write timestamp; anything written after the decision date is out |
| **Duplicate / group** | the same applicant applying several times, or an agency submitting near-identical CVs for many roles | group by applicant and by CV near-duplicate (hash the text); check overlap across the split |
| **Temporal** | training on this year's hires to predict last year's, across a change in hiring criteria or headcount | split chronologically by application date and compare with a random split |
| **Preprocessing** | fitting a text vectoriser, or selecting vocabulary terms, on all CVs before splitting - severe here, because text gives thousands of candidate features | move the vectoriser and any selection inside the pipeline and re-score |

The preprocessing row is the most dangerous of the four in this setting, precisely because text produces
wide data - and E5's arithmetic says a wide search finds impressive correlations in noise.

Worth adding, though it is not leakage: a model trained on past hiring decisions learns past hiring
*behaviour*, including any bias in it. That is a separate and serious problem, and it is module 13.

## Explain it to someone non-technical

### E19

> "Imagine setting an exam, and by accident the answer key gets photocopied onto the back of the question
> paper. Everyone scores 95% and you conclude the teaching was excellent. That is what happened to our
> model: one of the columns we fed it quietly contained the thing we were asking it to predict. The reason
> nobody notices is that the results look wonderful - there is no error message, just an unusually good
> number. It only shows up when the model meets a real case, where the answer is not printed on the back."

(87 words.)

## Optional challenge

### E20 - a leak detector that does not know what it is looking for

In [ ]:
def single_feature_scan(features, target, threshold=0.9):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
    full = cross_val_score(model, features, target, cv=folds, scoring="roc_auc").mean()
    rows = []
    for column in features.columns:
        alone = cross_val_score(model, features[[column]], target, cv=folds, scoring="roc_auc").mean()
        rows.append({"feature": column, "alone": round(alone, 4),
                     "share of the full model": round(alone / full, 3),
                     "flag": "SUSPICIOUS" if alone / full > threshold else ""})
    print("full model: %.4f" % full)
    return pd.DataFrame(rows).sort_values("alone", ascending=False)


print("-- 04-01's leaky churn table")
print(single_feature_scan(leaky_churn, churn).to_string(index=False))

print()
print("-- the noise data, with selection done leakily beforehand")
leaked_columns = pd.DataFrame(SelectKBest(f_classif, k=20).fit_transform(noise, coin_flip),
                              columns=["chosen_%d" % j for j in range(20)])
scan = single_feature_scan(leaked_columns, coin_flip)
print(scan.head(4).to_string(index=False))
print("... %d columns in total, %d flagged" % (len(scan), (scan.flag == "SUSPICIOUS").sum()))

**What it catches:** both whole-history columns on the churn table - `months_on_file` at 0.9323 and
`visits_lifetime` at 0.9153 - flagged without anyone having to suspect either in advance, and cleanly
separated from the legitimate features at 0.71 to 0.73. That is the value of the scan: it needs no
hypothesis, so it can find the leak you did not think of, and unlike the drop-one report of E12 it is not
confused by the two leaks covering for each other.

**What it misses:** the noise data. Not one of the 20 pre-selected columns comes close to the full model
on its own, because the leak there is **distributed** - twenty columns each carrying a small illegitimate
association, which only becomes a strong signal when combined. A per-column scan is blind to it by
construction, exactly as 04-01's per-column detector was blind to the `months_on_file` + `tenure_months`
pair.

**And why the pattern is suspicious rather than conclusive.** "One feature is nearly as good as all of
them" has a perfectly innocent explanation: some problems really are dominated by one variable. Predicting
house price from floor area, or tomorrow's temperature from today's, will trip this detector honestly.

So the scan is a **triage tool, not a verdict**. It tells you which column to interrogate, and then the
interrogation is the same two questions as always - *would this value have been known at prediction time,
and did this step look at `y`?*

Which is the honest summary of this whole chapter: **there is no automatic leakage detector, because
leakage is a statement about how the data will be available in the future, and no amount of staring at the
present tells you that.** The detectors narrow the search. The framing answers it.